# Salary-Cap Draft Optimizer — Reference Notebook

This notebook is the rebuilt, working replacement for the legacy
`NU_MSDS460_FFBSalaryCapOptimizer.ipynb`. The optimization logic now lives in
[`optimizer.py`](optimizer.py) so it can be tested and reused; this notebook is the
narrative demo. For live drafts, use the **web app** (`index.html`) — it re-optimizes
after every pick. See [`MODELING.md`](MODELING.md) for the evaluation of the original
model and the upgraded methodology.

**What was fixed/added vs the legacy notebook**
- The multi-lineup loop crashed with a `TypeError` (objective constrained against an
  uninitialized `list`). Alternatives are now generated with **no-good cuts**.
- **FLEX** slot, **bench budget reserve**, and graceful handling of missing positions.
- Points recomputed from stat-level projections under **your league's scoring**.
- **VORP** and intrinsic dollar values; a **max-bid advisor** for auction discipline.
- No `eval()` on solver output.

In [1]:
import pandas as pd
import plotly.express as px

import optimizer as opt

## 1. League settings

Configure your league here — everything downstream reacts to it.

In [2]:
league = opt.LeagueSettings(
    salary_cap=200,
    teams=12,
    slots={"QB": 1, "RB": 2, "WR": 2, "TE": 1, "FLEX": 1, "K": 1, "Def": 1},
    bench_spots=6,          # bench players still cost at least $1 each…
    bench_reserve_per_spot=1,  # …so the optimizer holds this back per bench spot
)
print(f"Optimizer budget for starters: ${league.optimizer_budget}")

Optimizer budget for starters: $194


## 2. Load player data and rescore it

`auc_values_ALL.xlsx` is the bundled legacy sheet (2022 vintage — swap in a current
export before your draft; the web app's Data tab makes this a drag-and-drop). Points
for skill positions are **recomputed from stat-level projections** under the scoring
you choose (`"ppr"`, `"half_ppr"`, `"standard"`, or a custom dict).

In [3]:
players = opt.load_players("auc_values_ALL.xlsx", scoring="ppr")
players = opt.add_vorp(players, league)
players[["Name", "Team", "Pos", "Value", "Pts", "VORP", "FairValue"]].head(10)

,Name,Team,Pos,Value,Pts,VORP,FairValue
0,Jonathan Taylor,IND,RB,72,315.0,119.0,66
1,Derrick Henry,TEN,RB,63,278.7,82.7,46
2,Dalvin Cook,MIN,RB,55,324.3,128.3,71
3,Nick Chubb,CLE,RB,54,278.9,82.9,46
4,Christian McCaffrey,CAR,RB,54,361.3,165.3,91
5,Joe Mixon,CIN,RB,53,265.7,69.7,39
6,Austin Ekeler,LAC,RB,51,270.5,74.5,41
7,Najee Harris,PIT,RB,50,271.5,75.5,42
8,Javonte Williams,DEN,RB,49,208.5,12.5,8
9,Leonard Fournette,TB,RB,47,171.1,-24.9,1


`VORP` is points above the last starter-quality player at the position;
`FairValue` distributes the league's discretionary cap in proportion to positive
VORP — the model's intrinsic price, to compare against the market `Value`.

## 3. Value vs. points

Each dot is a player (QB/RB/WR/TE — kickers and defenses are minimum-bid players and
only add noise here). Dots far **above** the pack at their price are bargains; the
table view in section 4 is the precise version of this picture.

In [4]:
POS_COLORS = {"QB": "#2a78d6", "RB": "#008300", "WR": "#e87ba4", "TE": "#eda100"}
skill = players[players.Pos.isin(POS_COLORS)]
fig = px.scatter(
    skill, x="Value", y="Pts", color="Pos",
    color_discrete_map=POS_COLORS,
    hover_name="Name", hover_data={"Team": True, "VORP": True, "FairValue": True},
    labels={"Value": "Auction value ($)", "Pts": "Projected points (PPR)"},
    title="Market price vs. projection — 2022 legacy sheet",
    template="simple_white",
)
fig.update_traces(marker={"size": 8, "opacity": 0.75,
                          "line": {"width": 1, "color": "#fcfcfb"}})
fig.update_layout(font_family="system-ui", legend_title_text="",
                  paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb")
fig.show()

## 4. Optimal lineups

The ILP maximizes projected points subject to the cap and slot structure. Alternative
lineups are *genuinely different rosters* (each previous roster is excluded with a
no-good cut), not score-perturbed repeats.

In [5]:
lineups = opt.optimize_lineups(players, league, n_lineups=5)
for n, lu in enumerate(lineups, 1):
    print(f"Lineup {n}: {lu['total_pts']} pts for ${lu['total_cost']}")
lineups[0]["players"]

Lineup 1: 2420.3 pts for $186
Lineup 2: 2419.3 pts for $186
Lineup 3: 2419.3 pts for $186
Lineup 4: 2418.3 pts for $185
Lineup 5: 2417.3 pts for $185


,Name,Team,Pos,Value,Pts
121,Evan McPherson,CIN,K,3,135.0
48,Patrick Mahomes,KC,QB,23,396.8
4,Christian McCaffrey,CAR,RB,54,361.3
22,Ezekiel Elliott,DAL,RB,35,309.7
73,Darren Waller,LV,TE,16,260.5
34,Davante Adams,LV,WR,27,326.2
32,Tyreek Hill,MIA,WR,27,324.3
420,Calvin Ridley,ATL,WR,1,306.5


## 5. Max-bid advisor

The number that should discipline your bidding: the largest price at which winning
the player still beats the best roster you could build without them. `None` means
the player doesn't make your optimal roster even at $1 — pass at any price.

In [6]:
for name in ["Jonathan Taylor", "Patrick Mahomes", "Cooper Kupp"]:
    print(opt.max_bid(players, name, league))

{'player': 'Jonathan Taylor', 'max_bid': 44, 'lineup_pts_without': 2420.3}


{'player': 'Patrick Mahomes', 'max_bid': 33, 'lineup_pts_without': 2413.8}


{'player': 'Cooper Kupp', 'max_bid': None, 'lineup_pts_without': 2420.3}


## 6. Mid-draft re-optimization

Lock in what you've actually drafted (at real prices), exclude players other teams
took, shrink the budget, and re-solve. This is exactly what the web app does
automatically on every pick.

In [7]:
my_picks = {"Christian McCaffrey": 61, "Davante Adams": 38}   # name -> price paid
gone = ["Jonathan Taylor", "Tyreek Hill", "Travis Kelce"]        # drafted by others

pool = players.copy()
locked = [pool.index[pool.Name == n][0] for n in my_picks]
for n, price in my_picks.items():
    pool.loc[pool.Name == n, "Value"] = price
excluded = [pool.index[pool.Name == n][0] for n in gone]

completion = opt.optimize_lineups(pool, league, locked=locked, excluded=excluded)[0]
print(f"Best completable roster: {completion['total_pts']} pts, "
      f"${completion['total_cost']} of ${league.optimizer_budget}")
completion["players"]

Best completable roster: 2389.9 pts, $193 of $194


,Name,Team,Pos,Value,Pts
121,Evan McPherson,CIN,K,3,135.0
82,Kyler Murray,ARI,QB,14,381.8
4,Christian McCaffrey,CAR,RB,61,361.3
22,Ezekiel Elliott,DAL,RB,35,309.7
73,Darren Waller,LV,TE,16,260.5
34,Davante Adams,LV,WR,38,326.2
41,D.K. Metcalf,SEA,WR,25,308.9
420,Calvin Ridley,ATL,WR,1,306.5


---
**Next step:** open `index.html` in a browser (or the hosted page) for the live
draft companion — same math, plus consensus blending, news boosts/fades, live
auction-inflation adjustment, and nomination strategy.